# Klasifikasi Lesi Kulit ISIC2018 Task 3 — CNN dengan Residual Connection (ResNet)

Notebook ini diterjemahkan dari `isic2018_resnet_pipeline.py` (Mini Project Komputasi Lunak — Arsitektur 2).

Alur pipeline:
1. **Configuration** — semua path dataset & hyperparameter dikumpulkan di satu section awal untuk mudah diubah.
2. Memuat ground truth resmi ISIC 2018 Task 3 & mengonversi *one-hot* ke label tunggal.
3. **EDA** — distribusi kelas, ukuran/channel gambar, statistik piksel per channel.
4. *Stratified split* train/validation dari training set resmi (test & validation resmi disisihkan penuh untuk evaluasi akhir).
5. Balancing opsional via downsampling kelas mayoritas.
6. Augmentasi + normalisasi ImageNet, dataset & DataLoader.
7. Model **ResNet configurable**: depth (18/34/50/101), jenis residual block, base channels, classifier hidden dim — bobot pretrained ImageNet dipakai otomatis bila kombinasinya standar.
8. Satu cell training (baseline) yang dipakai untuk SEMUA konfigurasi: eksperimen dilakukan dengan mengubah nilai hyperparameter di Section 1 + ganti `run_name` (hardcoded), lalu jalankan ulang cell yang sama — 1 eksperimen = 1 model = 1 run; ringkasan dari `runs_log.csv`.
9. Evaluasi di validation set resmi (ground truth resmi); setiap run mencatat skema training-test-eval & seluruh konfigurasi, dan menyimpan metrik, confusion matrix, serta sampel salah klasifikasi.

> Struktur data mengikuti rilis resmi ISIC 2018 Challenge (bukan versi Kaggle). Validation_Input resmi tidak memiliki ground truth publik sehingga tidak dipakai untuk evaluasi.

## 1. Configuration — path dataset & semua hyperparameter

> **Ubah nilai di sini.** Semua path **terdeteksi otomatis**: di Kaggle diambil dari `/kaggle/input` (folder yang memuat `ISIC2018_Task3_Training_Input` dipilih otomatis), di lokal memakai folder `dataset/`. Semua output (model, history, laporan, gambar) **disimpan** ke `CONFIG['output_dir']`.

In [ ]:
# =========================================================================
# SECTION 1: CONFIGURATION (SEMUA hyperparameter & path DIKUMPULKAN DI SINI)
# =========================================================================
import os
import glob
from pathlib import Path

CONFIG = {
    # ================= PATH DATASET =================
    # Dataset mengikuti rilis resmi ISIC 2018 Task 3.
    # - KAGGLE : path otomatis dari /kaggle/input (folder berisi
    #            ISIC2018_Task3_Training_Input dipilih otomatis).
    # - LOKAL  : pakai folder relatif "dataset".
    # Boleh juga eksplisit, misal "/kaggle/input/isic2018-task3".
    "data_dir": "dataset",

    # Nama subfolder/CSV relatif di dalam data_dir (mengikuti rilis resmi).
    "train_img_dir": "ISIC2018_Task3_Training_Input",
    "test_img_dir":  "ISIC2018_Task3_Test_Input",
    "val_img_dir":   "ISIC2018_Task3_Validation_Input",
    "train_gt_dir":  "ISIC2018_Task3_Training_GroundTruth",
    "test_gt_dir":   "ISIC2018_Task3_Test_GroundTruth",
    "val_gt_dir":    "ISIC2018_Task3_Validation_GroundTruth",

    # ================= OUTPUT (disimpan ke folder ini) =================
    # Di Kaggle cwd = /kaggle/working. Semua hasil (format file) tersimpan.
    "output_dir": "output",
    "save_model": True,     # simpan checkpoint state_dict model terbaik per run
    "save_history": True,   # simpan history loss/acc per run (JSON)

    # ================= DATA =================
    "class_columns": ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"],
    "num_classes": 7,
    "loss_weight": [],          # bobot MANUAL per kelas utk CrossEntropyLoss (urutan class_columns: [MEL, NV, BCC, AKIEC, BKL, DF, VASC]); [] = serahkan ke loss_weight_mode
    "loss_weight_mode": "inverse_frequency",  # bobot otomatis: 'none' (tanpa bobot) | 'inverse_frequency' (dihitung 1/frekuensi kelas dari distribusi training ASLI, relatif antar kelas); diabaikan bila loss_weight manual terisi
    "img_size": 224,            # ukuran input gambar sebelum masuk model (harus 224 utk ResNet); opsi nilai: 128 | 224 | 299
    "val_ratio": 0.15,          # proporsi training set untuk validation selama training (split stratified); opsi nilai: 0.10 | 0.15 | 0.20
    "random_state": 42,         # seed untuk reproduksibilitas split & sampling acak; opsi nilai: 42 | 123 | 2024
    "balance_threshold": 500,   # ambang downsampling kelas mayoritas (0 = data tidak di-balance); opsi nilai: 250 | 500 | 1000 | 0
    "sample_size": 300,         # jumlah gambar acak yang dianalisis di EDA; opsi nilai: 150 | 300 | 500

    # ================= IMAGE NORMALIZATION (pretrained ImageNet) =================
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std":  [0.229, 0.224, 0.225],

    # ================= TRAINING (baseline) =================
    "batch_size": 32,             # jumlah gambar yang diproses per langkah optimasi; opsi nilai: 8 | 16 | 32 | 64
    "lr": 1e-4,                   # learning rate, besar langkah update bobot; opsi nilai: 1e-5 | 1e-4 | 3e-4 | 1e-3
    "dropout": 0.3,               # probabilitas neuron dimatikan di classifier head (regularisasi); opsi nilai: 0.0 | 0.2 | 0.3 | 0.5
    "optimizer_name": "adam",     # algoritma optimasi bobot; opsi nilai: 'adam' | 'sgd' | 'rmsprop'
    "weight_decay": 1e-4,         # regularisasi L2 (penalti bobot besar); opsi nilai: 0.0 | 1e-5 | 1e-4 | 1e-3
    "num_epochs": 20,             # berapa kali seluruh training set dilihat model; opsi nilai: 10 | 20 | 30

    # ================= MODEL ARSITEKTUR (ResNet, nilai tunggal & diubah manual) =================
    # Backbone diterjemahkan menjadi ResNet dengan kedalaman/block/channel tertentu.
    # Bobot pretrained ImageNet dipakai OTOMATIS hanya jika kombinasi depth-block-channel
    # sesuai arsitektur standar (18/34=basic, 50/101=bottleneck, base_channels=64);
    # kombinasi lain dibangun dari nol tanpa pretrained.
    "depth": 18,                 # kedalaman backbone ResNet (semakin dalam makin besar); opsi nilai: 18 | 34 | 50 | 101
    "residual_blocks": "basic",  # jenis blok residual backbone; opsi nilai: 'basic' | 'bottleneck'
    "base_channels": 64,         # jumlah channel awal backbone (conv1); opsi nilai: 32 | 64 | 128
    "classifier_hidden_dim": 512,  # dimensi hidden layer classifier head (0 = tanpa hidden layer); opsi nilai: 0 | 128 | 256 | 512 | 1024
    "use_pretrained": True,      # pakai bobot pretrained ImageNet (hanya bila kombinasi standar); opsi nilai: True | False

    # ================= LINGKUNGAN =================
    "device": "auto",           # perangkat untuk training & evaluasi; opsi nilai: 'auto' | 'cuda' | 'cpu'
    "num_workers": 0,           # jumlah proses paralel pemuat data; opsi nilai: 0 | 2 | 4 (0 aman utk Windows)
    "pin_memory": True,         # salin batch ke pinned memory agar transfer ke GPU lebih cepat; opsi nilai: True | False
}

# ---- Resolusi path: otomatis cari dataset di /kaggle/input, selain itu pakai data_dir ----
OUTPUT_DIR    = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _resolve_data_root(configured):
    # Utamakan path eksplisit. Di Kaggle, cari otomatis folder /kaggle/input
    # yang memuat ISIC2018_Task3_Training_Input.
    explicit = str(configured)
    if os.path.isdir(os.path.join(explicit, CONFIG["train_img_dir"])):
        return explicit
    if os.path.isdir("/kaggle/input"):
        for entry in sorted(os.listdir("/kaggle/input")):
            cand = os.path.join("/kaggle/input", entry)
            if os.path.isdir(os.path.join(cand, CONFIG["train_img_dir"])):
                return cand
    return explicit

DATA_DIR = _resolve_data_root(CONFIG["data_dir"])

TRAIN_IMG_DIR = os.path.join(DATA_DIR, CONFIG["train_img_dir"])
TEST_IMG_DIR  = os.path.join(DATA_DIR, CONFIG["test_img_dir"])
VAL_IMG_DIR   = os.path.join(DATA_DIR, CONFIG["val_img_dir"])

# Ground truth berbentuk CSV di dalam folder *_GroundTruth.
def _first_csv(folder):
    hits = glob.glob(os.path.join(folder, "*.csv"))
    return hits[0] if hits else None

TRAIN_GT_PATH = _first_csv(os.path.join(DATA_DIR, CONFIG["train_gt_dir"]))
TEST_GT_PATH  = _first_csv(os.path.join(DATA_DIR, CONFIG["test_gt_dir"]))
VAL_GT_PATH   = _first_csv(os.path.join(DATA_DIR, CONFIG["val_gt_dir"]))

for p, desc in [(TRAIN_IMG_DIR, "Training input"), (TEST_IMG_DIR, "Test input"),
                (VAL_IMG_DIR, "Validation input")]:
    assert os.path.isdir(p), f"{desc}: folder tidak ditemukan -> {p}"
for p, desc in [(TRAIN_GT_PATH, "Training ground truth"), (TEST_GT_PATH, "Test ground truth"),
                (VAL_GT_PATH, "Validation ground truth")]:
    assert p and os.path.isfile(p), f"{desc}: file tidak ditemukan -> {p}"

IMG_SIZE  = CONFIG["img_size"]
CLASS_COLUMNS = CONFIG["class_columns"]

print("Data directory   :", DATA_DIR)
print("Output directory :", OUTPUT_DIR)
print("Train gambar     :", len(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg"))))
print("Test gambar      :", len(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))))
print("Validation gambar:", len(glob.glob(os.path.join(VAL_IMG_DIR, "*.jpg"))))
print("Semua path dataset valid. Config dimuat; DEVICE di-set di Section 2 (setelah import torch).")

## 2. Imports

In [ ]:
# =========================================================================
# SECTION: IMPORTS
# =========================================================================
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# =========================================================================
# DEVICE (dipindah ke sini karena butuh torch)
# =========================================================================
if CONFIG["device"] == "cpu":
    device = torch.device("cpu")
elif CONFIG["device"] == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA diminta tapi tidak tersedia.")
    device = torch.device("cuda")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device yang dipakai:", device)

print("PyTorch", torch.__version__, "| device:", device)
print("Data train:", len(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.jpg"))),
      "| Data test:", len(glob.glob(os.path.join(TEST_IMG_DIR, "*.jpg"))))

## 3. Muat Ground Truth (one-hot → label tunggal)

In [ ]:
# =========================================================================
# SECTION 2: Konversi Ground Truth One-Hot ke Label Tunggal
# Ground truth resmi ISIC2018 berformat one-hot (kolom MEL, NV, BCC,
# AKIEC, BKL, DF, VASC, isinya 1.0 di kolom kelas yang benar).
# =========================================================================

def onehot_to_label(df_raw):
    df = df_raw.copy()
    df["dx"] = df[CLASS_COLUMNS].idxmax(axis=1)
    return df[["image", "dx"]]


train_gt_raw = pd.read_csv(TRAIN_GT_PATH)
test_gt_raw  = pd.read_csv(TEST_GT_PATH)
val_gt_raw   = pd.read_csv(VAL_GT_PATH)
print("Contoh ground truth training:")
print(train_gt_raw.head())

train_df_full = onehot_to_label(train_gt_raw)
test_df  = onehot_to_label(test_gt_raw)
val_df_official = onehot_to_label(val_gt_raw)

train_df_full["path"] = train_df_full["image"].apply(lambda x: os.path.join(TRAIN_IMG_DIR, x + ".jpg"))
test_df["path"]       = test_df["image"].apply(lambda x: os.path.join(TEST_IMG_DIR, x + ".jpg"))
val_df_official["path"] = val_df_official["image"].apply(lambda x: os.path.join(VAL_IMG_DIR, x + ".jpg"))

print("\nJumlah data training (sebelum split val):", len(train_df_full))
print("Jumlah data test:", len(test_df))
print("Jumlah data validation resmi (berlabel):", len(val_df_official))

## 4. EDA — Distribusi Kelas

In [ ]:
# =========================================================================
# SECTION 3: EDA — Class Distribution
# =========================================================================

def class_distribution_table(df, name):
    counts = df["dx"].value_counts()
    pct = df["dx"].value_counts(normalize=True) * 100
    table = pd.DataFrame({"Jumlah": counts, "Persentase (%)": pct.round(2)})
    table.index.name = f"Kelas ({name})"
    return table


train_dist = class_distribution_table(train_df_full, "Training")
test_dist  = class_distribution_table(test_df, "Test")
val_dist   = class_distribution_table(val_df_official, "Validation Resmi")

print("Distribusi kelas — Training set:")
print(train_dist)
print("\nDistribusi kelas — Test set:")
print(test_dist)
print("\nDistribusi kelas — Validation set resmi:")
print(val_dist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
train_df_full["dx"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Distribusi Kelas — Training Set")
axes[0].set_xlabel("Kelas")
axes[0].set_ylabel("Jumlah Gambar")

test_df["dx"].value_counts().plot(kind="bar", ax=axes[1], color="indianred")
axes[1].set_title("Distribusi Kelas — Test Set")
axes[1].set_xlabel("Kelas")
axes[1].set_ylabel("Jumlah Gambar")
plt.tight_layout()
plt.show()

## 5. EDA — Karakteristik Data (ukuran, channel, statistik piksel)

In [ ]:
# =========================================================================
# SECTION 4: EDA — Data Characteristics
# Dihitung dari sampel acak biar cepat (naikkan CONFIG["sample_size"] kalau
# waktu memungkinkan). Mengonfirmasi bahwa gambar tidak seragam ukurannya
# sehingga resize wajib dilakukan sebelum masuk model.
# =========================================================================

SAMPLE_SIZE = CONFIG["sample_size"]
sample_paths = train_df_full["path"].sample(n=SAMPLE_SIZE, random_state=CONFIG["random_state"]).tolist()

image_stats = []
for p in sample_paths:
    with Image.open(p) as img:
        width, height = img.size
        mode = img.mode          # 'RGB', 'L', 'RGBA', dst
        n_channels = len(img.getbands())
        file_size_kb = os.path.getsize(p) / 1024
    image_stats.append({
        "width": width, "height": height,
        "aspect_ratio": round(width / height, 3),
        "mode": mode, "n_channels": n_channels,
        "file_size_kb": round(file_size_kb, 1)
    })

stats_df = pd.DataFrame(image_stats)
print(stats_df.head())

print("\n=== Ukuran Gambar ===")
print(f"Resolusi unik: {stats_df[['width','height']].drop_duplicates().shape[0]} variasi dari {SAMPLE_SIZE} sampel")
print(f"Width  -> min: {stats_df['width'].min()}, max: {stats_df['width'].max()}, modus: {stats_df['width'].mode()[0]}")
print(f"Height -> min: {stats_df['height'].min()}, max: {stats_df['height'].max()}, modus: {stats_df['height'].mode()[0]}")

print("\n=== Channel Warna ===")
print(stats_df["mode"].value_counts())
print(f"Semua gambar RGB (3 channel)? {(stats_df['n_channels'] == 3).all()}")

print("\n=== Descriptive Statistics (width, height, aspect ratio, file size) ===")
print(stats_df[["width", "height", "aspect_ratio", "file_size_kb"]].describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(stats_df["width"], bins=20, color="steelblue", edgecolor="black")
axes[0].set_title("Distribusi Lebar Gambar (px)")
axes[0].set_xlabel("Width")

axes[1].hist(stats_df["height"], bins=20, color="seagreen", edgecolor="black")
axes[1].set_title("Distribusi Tinggi Gambar (px)")
axes[1].set_xlabel("Height")

axes[2].hist(stats_df["file_size_kb"], bins=20, color="indianred", edgecolor="black")
axes[2].set_title("Distribusi Ukuran File (KB)")
axes[2].set_xlabel("File Size (KB)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Statistik piksel per channel (subset kecil, baca penuh piksel lebih berat) ---
pixel_means, pixel_stds = [], []
for p in sample_paths[:100]:
    with Image.open(p) as img:
        arr = np.array(img.convert("RGB")) / 255.0
        pixel_means.append(arr.reshape(-1, 3).mean(axis=0))
        pixel_stds.append(arr.reshape(-1, 3).std(axis=0))

pixel_means = np.array(pixel_means)
pixel_stds  = np.array(pixel_stds)

pixel_stat_df = pd.DataFrame({
    "Channel": ["R", "G", "B"],
    "Mean": pixel_means.mean(axis=0).round(4),
    "Std":  pixel_stds.mean(axis=0).round(4)
})
print("\nStatistik pixel dataset (skala 0-1):")
print(pixel_stat_df)
print("\nStatistik ImageNet (dipakai untuk normalisasi karena pretrained):")
print(f"Mean: {CONFIG['imagenet_mean']}, Std: {CONFIG['imagenet_std']}")

## 6. Split Train / Validation & Mapping Label

In [ ]:
# =========================================================================
# SECTION 5: Split Train / Validation (dari Training set resmi)
# Test set resmi ISIC2018 (dengan ground truth) disisihkan penuh untuk
# evaluasi akhir, tidak disentuh selama development.
# =========================================================================

train_classes = sorted(train_df_full["dx"].unique())
class_to_idx = {c: i for i, c in enumerate(train_classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}

train_df_full["label"] = train_df_full["dx"].map(class_to_idx)
test_df["label"]       = test_df["dx"].map(class_to_idx)
val_df_official["label"] = val_df_official["dx"].map(class_to_idx)

train_df, val_df = train_test_split(
    train_df_full, test_size=CONFIG["val_ratio"],
    stratify=train_df_full["label"], random_state=CONFIG["random_state"]
)

print("Mapping kelas:", class_to_idx)
print("Jumlah data train:", len(train_df))
print("Jumlah data validation:", len(val_df))
print("Jumlah data test (resmi):", len(test_df))

## 7. Balancing Data (opsional, mengikuti strategi paper)

In [ ]:
# =========================================================================
# SECTION 6: Data Balancing (opsional)
# Threshold dari CONFIG["balance_threshold"] (default 500): downsampling
# kelas mayoritas (NV, MEL, BKL). Kelas minoritas (AKIEC, BCC, VASC, DF)
# diperkuat lewat augmentasi saat training, bukan oversampling di sini.
# =========================================================================

BALANCE_THRESHOLD = CONFIG["balance_threshold"]


def downsample_majority(df, threshold, seed):
    balanced_parts = []
    for cls, group in df.groupby("dx"):
        if len(group) > threshold:
            group = group.sample(n=threshold, random_state=seed)
        balanced_parts.append(group)
    return pd.concat(balanced_parts).reset_index(drop=True)


train_df_balanced = downsample_majority(train_df, BALANCE_THRESHOLD, CONFIG["random_state"])

print("Distribusi kelas training SEBELUM downsampling:")
print(train_df["dx"].value_counts())
print("\nDistribusi kelas training SETELAH downsampling:")
print(train_df_balanced["dx"].value_counts())

## 8. Transform & Augmentasi

In [ ]:
# =========================================================================
# SECTION 7: Transform dan Augmentasi
# Resize ke CONFIG["img_size"] (224x224, input ResNet-18), normalisasi pakai
# statistik ImageNet karena pakai pretrained weights. Augmentasi hanya
# diterapkan ke data training.
# =========================================================================

IMAGENET_MEAN = CONFIG["imagenet_mean"]
IMAGENET_STD  = CONFIG["imagenet_std"]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## 9. Custom Dataset

In [ ]:
# =========================================================================
# SECTION 8: Custom Dataset
# =========================================================================

class ISICDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        label = row["label"]
        if self.transform:
            image = self.transform(image)
        return image, label

## 10. Model ResNet (configurable: depth, block, channel, classifier)

Backbone dibangun dari blok residual standar (*BasicBlock* / *Bottleneck*) sehingga `CONFIG['depth']`, `CONFIG['residual_blocks']`, `CONFIG['base_channels']`, dan `CONFIG['classifier_hidden_dim']` bisa diubah untuk eksperimen. Bobot pretrained ImageNet dipakai otomatis bila kombinasi arsitekturnya standar (18/34-basic, 50/101-bottleneck, 64 channel); kombinasi lain dibangun dari nol.

In [ ]:
# =========================================================================
# SECTION 9: Model ResNet configurable (depth, residual block, channel, classifier)
# =========================================================================

_DEPTH_LAYERS = {
    18: [2, 2, 2, 2], 34: [3, 4, 6, 3],
    50: [3, 4, 6, 3], 101: [3, 4, 23, 3],
}


class BasicBlock(nn.Module):
    # Blok residual dua konvolusi 3x3 (ResNet 18/34).
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels))
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out, inplace=True)


class BottleneckBlock(nn.Module):
    # Blok residual tiga konvolusi 1x1-3x3-1x1 (ResNet 50/101).
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride,
                               padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels * self.expansion))
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.bn3(self.conv3(out))
        out += self.shortcut(x)
        return F.relu(out, inplace=True)


class ResNetCustom(nn.Module):
    # ResNet dari nol dengan jumlah layer, jenis block, base channel, dan
    # classifier head yang bisa diatur lewat parameter.
    def __init__(self, block, layers, num_classes, base_channels=64,
                 classifier_hidden_dim=512, dropout=0.3):
        super().__init__()
        self.in_channels = base_channels
        self.conv1 = nn.Conv2d(3, base_channels, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(base_channels)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, base_channels,       layers[0], stride=1)
        self.layer2 = self._make_layer(block, base_channels * 2,   layers[1], stride=2)
        self.layer3 = self._make_layer(block, base_channels * 4,   layers[2], stride=2)
        self.layer4 = self._make_layer(block, base_channels * 8,   layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        in_features = base_channels * 8 * block.expansion
        if classifier_hidden_dim and classifier_hidden_dim > 0:
            head = [nn.Flatten(),
                    nn.Linear(in_features, classifier_hidden_dim),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p=dropout),
                    nn.Linear(classifier_hidden_dim, num_classes)]
        else:
            head = [nn.Flatten(), nn.Dropout(p=dropout),
                    nn.Linear(in_features, num_classes)]
        self.fc = nn.Sequential(*head)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        layers = []
        for s in [stride] + [1] * (num_blocks - 1):
            layers.append(block(self.in_channels, out_channels, stride=s))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(F.relu(self.bn1(self.conv1(x)), inplace=True))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.avgpool(x)
        return self.fc(x)


_BLOCK_BY_TYPE = {"basic": BasicBlock, "bottleneck": BottleneckBlock}


def build_model(num_classes=CONFIG["num_classes"], dropout=CONFIG["dropout"],
                depth=CONFIG["depth"], block_type=CONFIG["residual_blocks"],
                base_channels=CONFIG["base_channels"],
                classifier_hidden_dim=CONFIG["classifier_hidden_dim"],
                use_pretrained=CONFIG["use_pretrained"]):
    layers = _DEPTH_LAYERS.get(int(depth), [2, 2, 2, 2])
    block = _BLOCK_BY_TYPE[block_type]
    is_standard = (
        base_channels == 64 and
        ((block_type == "basic" and depth in (18, 34)) or
         (block_type == "bottleneck" and depth in (50, 101)))
    )
    use_pretrained = bool(use_pretrained) and is_standard

    if use_pretrained:
        arch_map = {
            18: torchvision.models.resnet18, 34: torchvision.models.resnet34,
            50: torchvision.models.resnet50, 101: torchvision.models.resnet101,
        }
        model = arch_map[int(depth)](weights="IMAGENET1K_V1")
        in_features = model.fc.in_features
        if classifier_hidden_dim and classifier_hidden_dim > 0:
            model.fc = nn.Sequential(
                nn.Flatten(),
                nn.Linear(in_features, classifier_hidden_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(p=dropout),
                nn.Linear(classifier_hidden_dim, num_classes))
        else:
            model.fc = nn.Sequential(nn.Flatten(), nn.Dropout(p=dropout),
                                     nn.Linear(in_features, num_classes))
        label_src = "torchvision pretrained ImageNet"
    else:
        model = ResNetCustom(block, layers, num_classes,
                             base_channels=base_channels,
                             classifier_hidden_dim=classifier_hidden_dim, dropout=dropout)
        label_src = "custom (dari nol)"
    model.pretrained_used = use_pretrained
    model.arch_label = (f"ResNet-{depth} [{block_type}, base={base_channels}, "
                        f"hidden={classifier_hidden_dim}] ({label_src})")
    return model.to(device)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# sanity check arsitektur
m = build_model()
tot, tr = count_params(m)
print(m.arch_label)
print(f"Param: {tot/1e6:.2f}M total, {tr/1e6:.2f}M trainable")

## 11. Fungsi Training & Evaluasi per Epoch

In [ ]:
# =========================================================================
# SECTION 10: Fungsi Training dan Evaluasi per Epoch
# =========================================================================

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, all_preds, all_labels

## 12. Fungsi Utama Training (hiperparameter sebagai argumen)

Setiap pemanggilan `run_training` mencatat **seluruh skema training-validation-test (resmi) sekaligus seluruh hiperparameter & konfigurasi** ke `run_<nama>.json` dan baris ringkasnya ke `runs_log.csv`. Model, history, dan skema tersimpan per run.

In [ ]:
# =========================================================================
# SECTION 11: Fungsi Utama Training + Pencatatan Run (skema & konfigurasi)
# =========================================================================

def _serializable_config():
    return {k: v for k, v in CONFIG.items() if not k.startswith("_")}


# Bobot efektif CrossEntropy yang sedang dipakai (untuk pencatatan run).
LOSS_WEIGHTS_USED = []


def build_criterion():
    # Loss CrossEntropyLoss. Bobot per kelas: manual (CONFIG["loss_weight"]) atau otomatis
    # inverse frequency dari distribusi training ASLI bila loss_weight_mode = 'inverse_frequency'
    # (diabaikan bila manual terisi). Konsisten untuk training & semua evaluasi;
    # bobot efektif disalin ke LOSS_WEIGHTS_USED agar bisa dicatat di run_<nama>.json.
    global LOSS_WEIGHTS_USED
    w = CONFIG["loss_weight"] or None
    if not w and CONFIG["loss_weight_mode"] == "inverse_frequency":
        counts = [int(train_df["dx"].value_counts().get(cls, 1)) for cls in train_classes]
        total = sum(counts)
        w = [total / (len(counts) * c) for c in counts]
    LOSS_WEIGHTS_USED = [float(x) for x in w] if w else []
    if w:
        w = torch.tensor(w, dtype=torch.float32, device=device)
    return nn.CrossEntropyLoss(weight=w)


def log_run_summary(row):
    # Append/update baris ringkas per run di runs_log.csv (re-run mengganti baris lama).
    log_path = OUTPUT_DIR / "runs_log.csv"
    rows = []
    if log_path.exists():
        rows = pd.read_csv(log_path).to_dict("records")
    rows = [r for r in rows if str(r["run"]) != str(row["run"])]
    rows.append(row)
    pd.DataFrame(rows).to_csv(log_path, index=False)
    print(f"  logged run -> {log_path}")


def run_training(lr=CONFIG["lr"], batch_size=CONFIG["batch_size"], dropout=CONFIG["dropout"],
                 optimizer_name=CONFIG["optimizer_name"], weight_decay=CONFIG["weight_decay"],
                 num_epochs=CONFIG["num_epochs"], run_name="baseline",
                 train_data=None,
                 depth=CONFIG["depth"], block_type=CONFIG["residual_blocks"],
                 base_channels=CONFIG["base_channels"],
                 classifier_hidden_dim=CONFIG["classifier_hidden_dim"],
                 use_pretrained=CONFIG["use_pretrained"]):

    if train_data is None:
        train_data = train_df_balanced

    train_loader = DataLoader(
        ISICDataset(train_data, train_transform),
        batch_size=batch_size, shuffle=True,
        num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"]
    )
    val_loader = DataLoader(
        ISICDataset(val_df, eval_transform),
        batch_size=batch_size, shuffle=False,
        num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"]
    )

    model = build_model(dropout=dropout, depth=depth, block_type=block_type,
                        base_channels=base_channels,
                        classifier_hidden_dim=classifier_hidden_dim,
                        use_pretrained=use_pretrained)
    criterion = build_criterion()
    if optimizer_name == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    elif optimizer_name == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Optimizer '{optimizer_name}' tidak dikenali")

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc, best_state = 0.0, None

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f"[{run_name}] Epoch {epoch+1}/{num_epochs} - "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    history["best_val_acc"] = best_val_acc
    tot_params, tr_params = count_params(model)

    # ---- Simpan model terbaik (val acc tertinggi) ----
    if CONFIG["save_model"] and best_state is not None:
        model_path = OUTPUT_DIR / f"model_{run_name}_best.pt"
        torch.save(best_state, str(model_path))
        print(f"  saved model {run_name} (val_acc {best_val_acc:.4f}) -> {model_path}")

    # ---- Rekam skema training-test-eval + seluruh konfigurasi run ----
    final = {
        "final_train_loss": history["train_loss"][-1],
        "final_train_acc":  history["train_acc"][-1],
        "final_val_loss":   history["val_loss"][-1],
        "final_val_acc":    history["val_acc"][-1],
        "best_val_acc":     best_val_acc,
        "test_resmi_metrics": "dievaluasi di Section 18",   # terisi saat evaluasi final
        "val_resmi_metrics": "dievaluasi di Section 19",
    }
    run_record = {
        "run_name": run_name,
        "scheme": {
            "phases": ["train", "validation (stratified split)", "test resmi", "validation resmi"],
            "train_samples": int(len(train_data)),
            "validation_split_samples": int(len(val_df)),
            "test_resmi_samples": int(len(test_df)),
            "validation_resmi_samples": int(len(val_df_official)),
            "validation_split_ratio": CONFIG["val_ratio"],
            "balancing": {"balance_threshold": CONFIG["balance_threshold"],
                          "train_after_balancing": int(len(train_data))},
            "random_state": CONFIG["random_state"],
            "device": str(device),
        },
        "hyperparameters": {
            "optimizer": optimizer_name, "learning_rate": lr,
            "batch_size": batch_size, "dropout": dropout,
            "weight_decay": weight_decay, "num_epochs": num_epochs,
        },
        "loss": {
            "function": "CrossEntropyLoss",
            "loss_weight_mode": CONFIG["loss_weight_mode"],
            "loss_weight_manual": CONFIG["loss_weight"],
            "loss_weight_applied": list(LOSS_WEIGHTS_USED),
        },
        "architecture": {
            "depth": depth, "residual_blocks": block_type,
            "base_channels": base_channels,
            "classifier_hidden_dim": classifier_hidden_dim,
            "pretrained_imageNet": bool(model.pretrained_used),
            "param_total": int(tot_params), "param_trainable": int(tr_params),
        },
        "dataset": {"source": "ISIC 2018 Task 3 (HAM10000)",
                    "classes": {cls: int(class_to_idx[cls]) for cls in train_classes}},
        "config": _serializable_config(),
        "history": history,
        "final_metrics": final,
    }
    if CONFIG["save_history"]:
        run_path = OUTPUT_DIR / f"run_{run_name}.json"
        with open(run_path, "w", encoding="utf-8") as f:
            json.dump(run_record, f, indent=2, ensure_ascii=False)
        print(f"  saved run record -> {run_path}")

    log_run_summary({
        "run": run_name,
        "experiment": "baseline" if run_name == "baseline" else "eksperimen",
        "depth": depth, "block_type": block_type,
        "base_channels": base_channels, "classifier_hidden_dim": classifier_hidden_dim,
        "pretrained": bool(model.pretrained_used),
        "dropout": dropout, "batch_size": batch_size, "lr": lr,
        "optimizer": optimizer_name, "epochs": num_epochs,
        "train_samples": int(len(train_data)),
        "val_split_samples": int(len(val_df)),
        "best_val_acc": round(best_val_acc, 4),
        "final_val_acc": round(history["val_acc"][-1], 4),
        "final_val_loss": round(history["val_loss"][-1], 4),
    })

    return model, history, run_record

## 13. Plot Kurva Loss & Accuracy

In [ ]:
# =========================================================================
# SECTION 13: Plot Kurva Loss dan Accuracy
# =========================================================================

def plot_history(history, title="Training History", fname=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train Loss")
    axes[0].plot(history["val_loss"], label="Val Loss")
    axes[0].set_title(f"{title} - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="Train Acc")
    axes[1].plot(history["val_acc"], label="Val Acc")
    axes[1].set_title(f"{title} - Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    if fname:
        fig.savefig(str(OUTPUT_DIR / fname), bbox_inches="tight", dpi=150)
        print(f"  saved figure -> {OUTPUT_DIR / fname}")
    plt.show()

## 14. Jalankan Training (1 model per konfigurasi)

Ini SATU-SATUNYA cell training. Setiap konfigurasi menghasilkan **1 model / 1 run**.

Cara melakukan **eksperimen** (bukan ablation):
1. Ubah nilai hyperparameter yang ingin dicoba **langsung di Section 1 (CONFIG)**,
2. Ganti `run_name` pada pemanggilan `run_training` di bawah ini secara **hardcoded** (mis. `run_name="lr_1e-3"`), ubah dari `"baseline"`,
3. Jalankan ulang cell ini saja.

Setiap eksperimen tersimpan sebagai run terpisah: `model_<run_name>_best.pt`, `run_<run_name>.json`, plus baris di `runs_log.csv` (re-run nama yang sama akan menggantikan baris lama). Contoh yang bisa dicoba diubah di CONFIG: LR, batch size, dropout, optimizer, weight decay, depth, residual blocks, base channels, classifier hidden dim.

In [ ]:
# =========================================================================
# SECTION 12: Jalankan Training (baseline / eksperimen)
# Baseline memakai setting default CONFIG sebagai patokan.
# Eksperimen: ubah CONFIG di Section 1 + ganti run_name hardcoded di bawah,
# lalu jalankan ulang cell ini. 1 eksperimen = 1 model = 1 run.
# =========================================================================

baseline_model, baseline_history, baseline_run_record = run_training(
    lr=CONFIG["lr"], batch_size=CONFIG["batch_size"], dropout=CONFIG["dropout"],
    optimizer_name=CONFIG["optimizer_name"], weight_decay=CONFIG["weight_decay"],
    num_epochs=CONFIG["num_epochs"], run_name="baseline"
)

plot_history(baseline_history, "Baseline", fname="history_baseline.png")

## 15. Eksperimen Hyperparameter (1 model per eksperimen)

Tidak ada cell eksperimen terpisah dan tidak ada ablation — cukup **cell Section 14** dengan kombinasi yang sama untuk semua eksperimen:

- **Eksperimen learning rate**: ubah `CONFIG['lr']` (opsi: `1e-5 | 1e-4 | 3e-4 | 1e-3`), set `run_name="lr_<nilai>"`, jalankan ulang Section 14.
- **Eksperimen batch size**: ubah `CONFIG['batch_size']` (opsi: `8 | 16 | 32 | 64`), set `run_name="bs_<nilai>"`, jalankan ulang Section 14.
- **Eksperimen dropout**: ubah `CONFIG['dropout']` (opsi: `0.0 | 0.2 | 0.3 | 0.5`), set `run_name="dropout_<nilai>"`, jalankan ulang Section 14.
- **Eksperimen optimasi** (opsional): ubah `CONFIG['optimizer_name']`/`CONFIG['weight_decay']`, set `run_name` sesuai, jalankan ulang Section 14.
- **Eksperimen arsitektur**: ubah `CONFIG['depth']` (`18 | 34 | 50 | 101`), `CONFIG['residual_blocks']` (`'basic' | 'bottleneck'`), `CONFIG['base_channels']` (`32 | 64 | 128`), atau `CONFIG['classifier_hidden_dim']` (`0 | 128 | 256 | 512 | 1024`); set `run_name` sesuai (mis. `depth_50`, `blocks_bottleneck`, `channels_32`, `hidden_256`), jalankan ulang Section 14.

Hasil perbandingan ditampilkan di Section 16 (dibaca dari `runs_log.csv`).

In [ ]:
# Argumen efektif yang akan dipakai run_training untuk eksperimen berikutnya
# (nilai diambil langsung dari CONFIG — ubah di Section 1).
current_run_cfg = {
    "run_name": "baseline",      # HARDCODED: ganti sesuai eksperimen (mis. "lr_1e-3")
    "lr": CONFIG["lr"],
    "batch_size": CONFIG["batch_size"],
    "dropout": CONFIG["dropout"],
    "optimizer": CONFIG["optimizer_name"],
    "weight_decay": CONFIG["weight_decay"],
    "epochs": CONFIG["num_epochs"],
    "depth": CONFIG["depth"],
    "residual_blocks": CONFIG["residual_blocks"],
    "base_channels": CONFIG["base_channels"],
    "classifier_hidden_dim": CONFIG["classifier_hidden_dim"],
    "use_pretrained": CONFIG["use_pretrained"],
    "loss_weight_mode": CONFIG["loss_weight_mode"],
    "loss_weight_manual": CONFIG["loss_weight"],
}
print(json.dumps(current_run_cfg, indent=2))

## 16. Ringkasan Hasil Eksperimen (tabel laporan)

Tabel di bawah dibaca dari `runs_log.csv` — berisi satu baris per run yang pernah dijalankan di Section 14 (baseline + semua eksperimen). Re-run dengan `run_name` yang sama akan menggantikan baris yang lama.

In [ ]:
# =========================================================================
# SECTION 15: Ringkasan Hasil Eksperimen (dibaca dari runs_log.csv)
# Setiap eksperimen = 1 run = 1 baris (baseline tetap 1 baris).
# =========================================================================

runs_log_path = OUTPUT_DIR / "runs_log.csv"
if runs_log_path.exists():
    runs_log_df = pd.read_csv(runs_log_path)
    print(runs_log_df)
else:
    print("runs_log.csv belum ada — jalankan Section 14 terlebih dahulu.")

## 17. Fungsi Evaluasi Lengkap (metrik, confusion matrix, sampel salah klasifikasi)

Satu helper dipakai untuk evaluasi di test set resmi dan validation set resmi. Untuk tiap dataset: metrik & classification report (CSV), confusion matrix (PNG), prediksi per-gambar (CSV), serta **sampel yang salah diklasifikasi** (CSV + grid gambar).

In [ ]:
# =========================================================================
# SECTION 16: Helper Evaluasi Lengkap + Sampel Salah Klasifikasi
# =========================================================================

criterion = build_criterion()


def evaluate_and_report(model, loader, df_ref, tag, title, cmap="Blues", max_samples=12):
    loss, acc, preds, labels = evaluate(model, loader, criterion)
    print(f"{title} -> Loss: {loss:.4f}, Accuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(labels, preds, target_names=train_classes))

    res = df_ref[["image", "dx", "path"]].copy()
    res.columns = ["image", "true_label", "path"]
    res["pred_class"] = [train_classes[p] for p in preds]
    res["correct"] = (res["true_label"] == res["pred_class"]).astype(int)

    # Metrik & laporan klasifikasi
    metrics_path = OUTPUT_DIR / f"{tag}_metrics.csv"
    pd.DataFrame({"loss": [loss], "accuracy": [acc]}).to_csv(metrics_path, index=False)
    report_path = OUTPUT_DIR / f"{tag}_classification_report.csv"
    pd.DataFrame(classification_report(labels, preds, target_names=train_classes,
                                       output_dict=True, zero_division=0)).T.to_csv(report_path)
    # Prediksi per-gambar
    res.to_csv(OUTPUT_DIR / f"{tag}_predictions.csv", index=False)
    print(f"saved {tag} metrics -> {metrics_path}")
    print(f"saved {tag} report -> {report_path}")
    print(f"saved {tag} predictions -> {OUTPUT_DIR / f'{tag}_predictions.csv'}")

    # Confusion matrix
    cm_fig = plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(labels, preds), annot=True, fmt="d",
                xticklabels=train_classes, yticklabels=train_classes, cmap=cmap)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {title}")
    plt.tight_layout()
    cm_path = OUTPUT_DIR / f"confusion_matrix_{tag}.png"
    cm_fig.savefig(str(cm_path), bbox_inches="tight", dpi=150)
    print(f"saved confusion matrix -> {cm_path}")
    plt.show()

    # Sampel salah klasifikasi (CSV + grid gambar)
    mis = res[res["correct"] == 0]
    mis_csv_path = OUTPUT_DIR / f"misclassified_{tag}.csv"
    mis.to_csv(mis_csv_path, index=False)
    print(f"\nSalah klasifikasi: {len(mis)} -> {mis_csv_path}")

    n = min(max_samples, len(mis))
    if n > 0:
        grid = int(np.ceil(np.sqrt(n)))
        fig_mis, axes = plt.subplots(grid, grid, figsize=(grid * 3, grid * 3))
        axes = np.array(axes).flatten()
        for i in range(n):
            im = Image.open(mis.iloc[i]["path"]).convert("RGB")
            axes[i].imshow(im)
            axes[i].set_title(f"{mis.iloc[i]['true_label']} -> {mis.iloc[i]['pred_class']}",
                              fontsize=9)
            axes[i].axis("off")
        for i in range(n, len(axes)):
            axes[i].axis("off")
        plt.suptitle(f"Misclassified - {title} (n={n})", fontsize=12)
        plt.tight_layout()
        mis_png_path = OUTPUT_DIR / f"misclassified_{tag}.png"
        fig_mis.savefig(str(mis_png_path), bbox_inches="tight", dpi=150)
        print(f"saved misclassified figure -> {mis_png_path}")
        plt.show()
    return loss, acc, preds, labels, res, mis

## 18. Evaluasi Akhir di Test Set Resmi

Pakai `baseline_model` atau model hasil kombinasi hyperparameter terbaik. Test set resmi ISIC2018 baru disentuh di sini — tidak pernah dilihat model selama training/tuning.

In [ ]:
# =========================================================================
# SECTION 17: Evaluasi Akhir di Test Set Resmi
# =========================================================================

# Gunakan model terbaik (mis. hasil eksperimen) sebagai best_model
best_model = baseline_model
criterion = build_criterion()

test_loader = DataLoader(
    ISICDataset(test_df, eval_transform), batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"]
)

test_loss, test_acc, _, _, test_res, test_mis = evaluate_and_report(
    best_model, test_loader, test_df, tag="test", title="Test Set Resmi ISIC2018", cmap="Blues"
)

## 19. Evaluasi di Validation Set Resmi

Validation ground truth resmi tersedia (193 gambar berlabel). Evaluasi model terbaik yang sama seperti pada test set, untuk melihat konsistensi performa di data validasi resmi (data yang dipisahkan panitia, bukan bagian dari training).

In [ ]:
# =========================================================================
# SECTION 18: Evaluasi di Validation Set Resmi (Ground Truth Resmi)
# =========================================================================

val_loader = DataLoader(
    ISICDataset(val_df_official, eval_transform), batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=CONFIG["pin_memory"]
)

val_loss, val_acc, _, _, val_res, val_mis = evaluate_and_report(
    best_model, val_loader, val_df_official, tag="validation",
    title="Validation Set Resmi ISIC2018", cmap="Purples"
)

## Catatan

- **Upload ke Kaggle**: attach dataset berisi folder standar ISIC 2018 Task 3 (input + ground truth). Path dataset **terdeteksi otomatis** dari `/kaggle/input`; jika struktur sama persis, tidak perlu ubah apa pun. Bila nama root dataset berbeda, langsung tambahkan `'/kaggle/input/<nama-dataset>'` ke `CONFIG['data_dir']`.
- **Pencatatan run**: setiap pemanggilan `run_training` (baseline & semua eksperimen) menyimpan **skema lengkap training-test-eval** (jumlah sampel train/val-split/test-resmi/val-resmi, rasio split, balancing, seed, device), **seluruh hiperparameter** (optimizer, lr, batch, dropout, weight decay, epoch), **arsitektur** (depth, block, channel, hidden dim, pretrained?, jumlah param), snapshot `CONFIG`, history, dan metrik akhir ke `run_<nama>.json`; baris ringkas untuk tiap run tercatat di **`runs_log.csv`** (re-run mengganti baris yang sama).
- **Semua output tersimpan** ke `CONFIG['output_dir']` (`output/`): model terbaik (`model_*_best.pt`), rekaman run (`run_*.json`, `runs_log.csv`), kurva training (PNG), dan hasil evaluasi di test set serta validation set resmi (metrics CSV, classification report CSV, confusion matrix PNG, prediksi per-gambar CSV, sampel salah klasifikasi CSV + PNG). Di Kaggle folder ini ada di `/kaggle/working/output/`.
- **Semua hyperparameter** (path, ukuran gambar, split, balancing, training, arsitektur) dikumpulkan di **Section 1 (CONFIG)** — ubah di satu tempat saja.
- **Eksperimen = 1 model, bukan ablation**: hanya ada satu cell training (Section 14). Untuk eksperimen, ubah nilai hyperparameter itu **langsung di Section 1** (mis. `CONFIG['lr']`, `CONFIG['depth']`) lalu ganti **`run_name` hardcoded** pada pemanggilan `run_training` (Section 14), lalu jalankan ulang cell itu. Satu eksperimen menghasilkan satu model/run; perbandingan dibaca dari `runs_log.csv` di Section 16.
- **Validation selama training** memakai stratified split dari training set (agar pembanding eksperimen konsisten). **Validation set resmi** (193 label) dievaluasi terpisah di **Section 19** memakai ground truth panitia.
- **Model**: ResNet dapat dikonfigurasi (depth 18/34/50/101, residual block basic/bottleneck, base channels, classifier hidden dim). Bobot pretrained ImageNet dipakai otomatis hanya untuk kombinasi arsitektur standar; kombinasi lain dibangun dari nol dengan kelas `ResNetCustom`.
- Section EDA menghasilkan class distribution, descriptive statistics (dimensi, ukuran file, statistik piksel per channel) — langsung bisa dipakai untuk laporan Metodologi.

## Referensi

Studi ini menerapkan klasifikasi **skin lesion** pada **ISIC 2018 Task 3** (7 kelas diagnosis: MEL, NV, BCC, AKIEC, BKL, DF, VASC). Dataset tugas ini merupakan bagian dari ISIC Challenge 2018 dan terkait langsung dengan HAM10000.

- [1] N. C. F. Codella, D. Gutman, M. E. Celebi, B. Helba, M. A. Marchetti, S. W. Dusza, A. Kalloo, K. Liopyris, N. Mishra, H. Kittler, A. Halpern, **"Skin Lesion Analysis Toward Melanoma Detection: A Challenge at the 2017 International Symposium on Biomedical Imaging (ISBI), Hosted by the International Skin Imaging Collaboration (ISIC)"**, 2018. arXiv:1710.05006.
- [2] N. Codella, V. Rotemberg, P. Tschandl, M. E. Celebi, S. Dusza, D. Gutman, B. Helba, A. Kalloo, K. Liopyris, M. Marchetti, H. Kittler, A. Halpern, **"Skin Lesion Analysis Toward Melanoma Detection 2018: A Challenge Hosted by the International Skin Imaging Collaboration (ISIC)"**, 2018. arXiv:1902.03368. https://arxiv.org/abs/1902.03368
- [3] P. Tschandl, C. Rosendahl, H. Kittler, **"The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions"**, Scientific Data 5, 180161 (2018). https://doi.org/10.1038/sdata.2018.161